<a href="https://colab.research.google.com/github/shilpamusale/codeexec-rl/blob/main/smoke_test_base_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%cd /content
!rm -rf codeexec-rl
!git clone -q https://github.com/shilpamusale/codeexec-rl.git
%cd /content/codeexec-rl
!pip install -q -e . --no-deps
!pip install -q evalplus==0.3.1 datasets

In [ ]:
import sys
sys.path.insert(0, "/content/codeexec-rl/src")
import codeexec
print("codeexec OK:", codeexec.__file__)

In [ ]:
# Load the model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-Coder-1.5B"   #base, not - Instruct (ADR-006)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="cuda")
model.eval()
print("loaded", MODEL, "on", model.device)

In [ ]:
# The subprocess grader (reuses build_harness)

import subprocess, sys
from codeexec.rewards.harness import build_harness, RESULT_SENTINEL
import json

def subprocess_grade(code: str, problem) -> bool:
    tests = [(tc.args, tc.expected) for tc in (*problem.visible_tests, *problem.heldout_tests)]
    harness = build_harness(code, problem.entry_point, tests, problem.atol)
    try:
        out = subprocess.run(
            [sys.executable, "-"],        # read program from STDIN, not -c
            input=harness,                # harness goes in via stdin (no length limit)
            capture_output=True, text=True, timeout=10,
        )
    except subprocess.TimeoutExpired:
        return False
    for line in reversed(out.stdout.splitlines()):
        if line.startswith(RESULT_SENTINEL):
            try:
                data = json.loads(line[len(RESULT_SENTINEL):].strip())
                return data["passed"] == data["total"] and data["total"] > 0
            except Exception:
                return False
    return False

In [ ]:
# Sample problems and run

import random
from codeexec.tasks.datasets import training_problems
from codeexec.eval.generate import make_generate_fn
from codeexec.eval.smoke import run_smoke_test

all_problems = list(training_problems().values())   # ~374 (slow load, ~1 min)
random.seed(0)
sample = random.sample(all_problems, 100)            # frozen sample of 100

generate_fn = make_generate_fn(model, tokenizer, max_new_tokens=256)

def progress(tid, malformed, passed):
    tag = "MALFORMED" if malformed else ("PASS" if passed else "fail")
    print(f"  {tid}: {tag}")

result = run_smoke_test(sample, generate_fn, subprocess_grade, on_problem=progress)

In [ ]:
# The two numbers + ADR verdict

print("=" * 50)
print(f"base pass@1:     {result.pass_at_1:.3f}  ({result.passed}/{result.total})")
print(f"malformed rate:  {result.malformed_rate:.3f}  ({result.malformed}/{result.total})")
print(f"well-formed:     {result.wellformed_rate:.3f}")
print("=" * 50)
print("ADR-001 (optimization pressure):",
      "OK — room to move" if 0.05 < result.pass_at_1 < 0.95 else "⚠ CHECK — too low/high")
print("ADR-006 (format reliability):",
      "OK — base stands" if result.malformed_rate < 0.15 else "⚠ HIGH — consider few-shot fallback")